In [1]:
import numpy as np
from sklearn.linear_model import RANSACRegressor
from sklearn.linear_model import LinearRegression
import cv2 

In [2]:
lines = np.genfromtxt('lines.csv', delimiter=',' , skip_header=1)

In [3]:
x1 = lines[:, 0]
y1 = lines[:, 3]

In [ ]:
points = np.column_stack((x1, y1))
centroid = points.mean(axis=0)
centered = points - centroid
U, S, Vt = np.linalg.svd(centered)
a, b = Vt[-1]
c = -(a * centroid[0] + b * centroid[1])

In [5]:
print(f"a = {a}")
print(f"b = {b}")
print(f"c = {c}")

a = 0.7735616496467872
b = -0.6337210539312553
c = -3.794192210845812


# Q1 Part b

In [6]:
X_cols = lines[:, :3]
Y_cols = lines[:, 3:]

X_all = X_cols.flatten().reshape(-1, 1)
Y_all = Y_cols.flatten()

remaining_X = X_all.copy()
remaining_Y = Y_all.copy()

In [7]:
lines = []
for i in range(3):
    ransac = RANSACRegressor(LinearRegression(), residual_threshold=0.5, random_state=42)
    ransac.fit(remaining_X, remaining_Y)
    
    m = ransac.estimator_.coef_[0]
    c = ransac.estimator_.intercept_
    lines.append((m, c))
    
    print(f"\nLine {i+1}")
    print(f"slope (m) = {m:.4f}")
    print(f"intercept (c) = {c:.4f}")
    
    inlier_mask = ransac.inlier_mask_
    remaining_X = remaining_X[~inlier_mask]
    remaining_Y = remaining_Y[~inlier_mask]


Line 1
slope (m) = -0.5269
intercept (c) = 1.9891

Line 2
slope (m) = 1.0436
intercept (c) = 0.9242

Line 3
slope (m) = 1.1266
intercept (c) = -6.4093


# Q 2

In [15]:
f = 8.0            
Z = 720.0          
pixel_size = 2.2e-3 

width_pixels = 120  
height_pixels = 150 

In [17]:
width_img_mm = width_pixels * pixel_size
height_img_mm = height_pixels * pixel_size

width_real = (Z / f) * width_img_mm
height_real = (Z / f) * height_img_mm

In [18]:
print(f"Real width ≈ {width_real:.2f} mm")
print(f"Real height ≈ {height_real:.2f} mm")

Real width ≈ 23.76 mm
Real height ≈ 29.70 mm


# q 3


In [8]:
points = []

In [9]:
def mouse_callback(event, x, y, flags, param):
    global points, img_display

    if event == cv2.EVENT_LBUTTONDOWN:
        if len(points) < 4:
            points.append((x, y))
            print(f"Point {len(points)}: ({x}, {y})")
            
            cv2.circle(img_display, (x, y), 5, (0, 0, 255), -1)
            cv2.imshow("Image", img_display)

        if len(points) == 4:
            print("\nFour points selected:")
            for i, p in enumerate(points):
                print(f"P{i+1}: {p}")
            print("Press any key to exit.")

In [10]:
img = cv2.imread("turf.jpg")
if img is None:
    raise FileNotFoundError("Image not found.")

img_display = img.copy()
cv2.namedWindow("Image")
cv2.setMouseCallback("Image", mouse_callback)

cv2.imshow("Image", img_display)
cv2.waitKey (0)
cv2.destroyAllWindows ()

points = np.array(points , dtype=np.float32)
print("\nFinal array of selected points:")
print(points)



Point 1: (386, 162)
Point 2: (597, 161)
Point 3: (378, 255)
Point 4: (1, 260)

Four points selected:
P1: (386, 162)
P2: (597, 161)
P3: (378, 255)
P4: (1, 260)
Press any key to exit.

Final array of selected points:
[[386. 162.]
 [597. 161.]
 [378. 255.]
 [  1. 260.]]


In [11]:
flag_img = cv2.imread("flag.png")

In [12]:
if len(points) == 4:
    h, w = flag_img.shape[:2]
    
    pts_src = np.array([
        [0, 0],          # Top-Left
        [w - 1, 0],      # Top-Right
        [w - 1, h - 1],  # Bottom-Right
        [0, h - 1]       # Bottom-Left
    ], dtype=np.float32)
    
    pts_dst = points
    
    matrix = cv2.getPerspectiveTransform(pts_src, pts_dst)
    warped_flag = cv2.warpPerspective(flag_img, matrix, (img.shape[1], img.shape[0]))
    
    mask = np.zeros(img.shape[:2], dtype=np.uint8)
    cv2.fillConvexPoly(mask, np.int32(pts_dst), 255)
    
    mask_inv = cv2.bitwise_not(mask)
    img_bg = cv2.bitwise_and(img, img, mask=mask_inv)
    
    flag_fg = cv2.bitwise_and(warped_flag, warped_flag, mask=mask)
    
    result = cv2.add(img_bg, flag_fg)

    # Display the final superimposed result
    cv2.imshow("Superimposed Result", result)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    
    